# G7B en Deepnote
En cada corrida se lee la orden publicada en main. El informe identifica el commit y la huella del cubo.
GitHub Actions aporta el cubo automáticamente. En el proyecto instala duckdb==2.0.0 y azure-storage-blob. AZURE_LAGO_URL como secreto de solo lectura permite descargar el cubo vigente. También puedes extraer el artefacto de Cubo G7B bajo insumo/ o configurar G7B_CUBO_DIR. Sin entrada el informe muestra el estado de GitHub y marca la consulta pendiente.


In [ ]:
# Instala las dependencias en el Python de este cuaderno.
%pip install -q "azure-storage-blob>=12.20,<13" "duckdb==2.0.0"
from azure.storage.blob import ContainerClient
import duckdb

In [ ]:
LAB = 'deepnote'
PERMITIDAS = {'captura_y_cubo', 'calidad_del_cubo'}
TAREA_ESTADO = 'captura_y_cubo'
WORKFLOWS = ['telemetria.yml', 'cubo.yml']
SQL = 'SELECT fuente, estado, mediciones FROM v_calidad ORDER BY fuente, estado'
import json, os, re
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

REPO = "Maximiliano-cabello-mata/amatista-g7b-etl"
def leer_json(url):
    req = Request(url, headers={"User-Agent": "G7B-laboratorio/1"})
    with urlopen(req, timeout=30) as respuesta:
        datos = respuesta.read(1_000_001)
    if len(datos) > 1_000_000:
        raise ValueError("Respuesta demasiado grande")
    return json.loads(datos)

api = "https://api.github.com/repos/" + REPO
commit = leer_json(api + "/branches/main")["commit"]["sha"]
if not re.fullmatch("[0-9a-f]{40}", commit):
    raise ValueError("Commit inesperado")
config = leer_json(f"https://raw.githubusercontent.com/{REPO}/{commit}/analisis/laboratorios/instrucciones.json")
if config.get("contrato") != "g7b.laboratorios/1":
    raise ValueError("Instrucciones incompatibles")
orden = config["laboratorios"][LAB]
if type(orden["activo"]) is not bool or not isinstance(orden["tareas"], list):
    raise ValueError("Orden inválida")
if not set(orden["tareas"]) <= PERMITIDAS:
    raise ValueError("Hay tareas desconocidas; actualiza el cuaderno")
informe = {"laboratorio": LAB, "commit": commit, "revision": config["revision"],
           "fecha_utc": datetime.now(timezone.utc).isoformat(), "activo": orden["activo"], "resultados": {}}

def ejecuciones(workflow):
    lista = leer_json(f"{api}/actions/workflows/{workflow}/runs?branch=main&per_page=10")["workflow_runs"]
    return [{"run": x["id"], "fecha": x["created_at"], "estado": x["status"],
             "resultado": x["conclusion"], "url": x["html_url"]} for x in lista]


In [ ]:
import hashlib
import tempfile
def cubo_azure(url):
    from azure.storage.blob import ContainerClient
    raiz = Path(tempfile.mkdtemp(prefix="g7b-laboratorio-"))
    with ContainerClient.from_container_url(url.strip()) as lago:
        puntero = json.loads(lago.download_blob("control/cubo_actual.json").readall())
        contenido = lago.download_blob(puntero["manifiesto"]).readall()
        if hashlib.sha256(contenido).hexdigest() != puntero["sha256"]:
            raise ValueError("Manifiesto remoto inconsistente")
        meta = json.loads(contenido)
        if meta.get("version") != puntero["version"] or meta.get("estado") != "verificado":
            raise ValueError("Versión remota no verificada")
        (raiz / "oro").mkdir()
        with (raiz / "oro/cubo.duckdb").open("wb") as archivo:
            lago.download_blob(meta["archivos"]["oro/cubo.duckdb"]["ruta"]).readinto(archivo)
        (raiz / "manifiesto.json").write_bytes(contenido)
    return raiz

def consultar_cubo():
    # El artefacto de Cubo G7B se puede extraer o subir a esta carpeta en cualquier laboratorio.
    raiz = Path(os.environ.get("G7B_CUBO_DIR", "insumo"))
    temporal = False
    if "G7B_CUBO_DIR" not in os.environ and not (raiz / "manifiesto.json").is_file() and os.environ.get("AZURE_LAGO_URL"):
        raiz = cubo_azure(os.environ["AZURE_LAGO_URL"])
        temporal = True
    manifiesto = raiz / "manifiesto.json"
    if not manifiesto.is_file():
        if "G7B_CUBO_DIR" in os.environ:
            raise FileNotFoundError("Falta manifiesto.json en G7B_CUBO_DIR")
        return {"estado": "sin_cubo_local", "detalle": "Consulta el informe de Actions o importa el artefacto de Cubo G7B."}
    meta = json.loads(manifiesto.read_text(encoding="utf-8"))
    archivo = raiz / "oro/cubo.duckdb"
    if meta.get("contrato") != "g7b.cubo/1":
        raise ValueError("Contrato de cubo inesperado")
    with archivo.open("rb") as f:
        digest = hashlib.sha256()
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(bloque)
    if digest.hexdigest() != meta["archivos"]["oro/cubo.duckdb"]["sha256"]:
        raise ValueError("El cubo no coincide con el manifiesto")
    import duckdb
    with duckdb.connect(str(archivo), read_only=True) as con:
        if con.execute("SELECT count(*) FROM mediciones").fetchone()[0] != meta["mediciones"]:
            raise ValueError("Conteo inconsistente")
        resultado = con.execute(SQL)
        columnas = [x[0] for x in resultado.description]
        salida = {"estado": "verificado", "sha256": digest.hexdigest(),
                  "filas": [dict(zip(columnas, fila)) for fila in resultado.fetchall()]}
    if temporal:
        archivo.unlink()
        manifiesto.unlink()
        (raiz / "oro").rmdir()
        raiz.rmdir()
    return salida

for tarea in orden["tareas"] if orden["activo"] else []:
    informe["resultados"][tarea] = ({w: ejecuciones(w) for w in WORKFLOWS}
                                    if tarea == TAREA_ESTADO else consultar_cubo())
ruta = Path(f"resultado_{LAB}.json")
ruta.write_text(json.dumps(informe, ensure_ascii=False, indent=2), encoding="utf-8")
print(ruta.read_text(encoding="utf-8"))
